# ETF Explorer (v1)

Discovery view over Stashaway's full ETF Explorer offering (~98 ETFs).
Computes multi-window metrics (1Y/3Y/5Y) + correlation with your combined book.
Renders to `reports/etf_explorer.html`.

In [1]:
from datetime import date
from pathlib import Path

from hailmary.allocation.book_config import MGMT_FEES_ANNUAL, ROLES
from hailmary.allocation.etf_explorer import build_etf_explorer, render_etf_explorer_report
from hailmary.allocation.portfolios import from_parsed
from hailmary.allocation.statements import parse_statement
from hailmary.allocation.returns import last_business_day_on_or_before
from hailmary.data.providers import YahooFinanceProvider

ETF_XLSX = Path('../../data/stashaway_etf_universe.xlsx')
STATEMENT_PATH = Path('../../data/statements/2026-04 StashAway Monthly Statement.pdf')
REPORT_PATH = Path('../../reports/etf_explorer.html')
START = date(2020, 1, 1)
END = last_business_day_on_or_before(date.today())
TARGET_ANN_RETURN = 0.05
print(f'window: {START}..{END}')

window: 2020-01-01..2026-05-25


## Load user's book (for correlation reference)

In [2]:
parsed = parse_statement(STATEMENT_PATH)
portfolios = [
    from_parsed(
        p,
        roles=ROLES[p.name],
        metadata={'management_fee_annual': MGMT_FEES_ANNUAL.get(p.name, 0.0)},
    )
    for p in parsed if p.name in ROLES
]
provider = YahooFinanceProvider()
fx_bars = provider.get_bars(['USDSGD=X'], START, END)
fx_series_usd_sgd = fx_bars.xs('USDSGD=X', level=0)['close']
print(f'{len(portfolios)} portfolios loaded')

2026-05-25 00:03:57.698 | DEBUG    | hailmary.allocation.statements:get:169 - Statement cache hit for 2026-04 StashAway Monthly Statement.pdf


2026-05-25 00:03:57.702 | DEBUG    | hailmary.data.cache:get:38 - Cache hit key=fa570043d3f4


15 portfolios loaded


## Build the explorer DataFrame

In [3]:
explorer_df = build_etf_explorer(
    ETF_XLSX,
    portfolios=portfolios,
    price_source=provider,
    fx_series_usd_sgd=fx_series_usd_sgd,
    start=START,
    end=END,
)
print(f'{len(explorer_df)} ETFs, {explorer_df["has_data"].sum()} with Yahoo data')
explorer_df.head(20)

2026-05-25 00:03:57.970 | INFO     | hailmary.allocation.etf_explorer:build_etf_explorer:167 - ETF Explorer: fetching 98 symbols from Yahoo…


2026-05-25 00:03:57.970 | INFO     | hailmary.data.providers.yahoo:get_bars:64 - Fetching 98 symbols from Yahoo (2020-01-01 → 2026-05-25); inclusive


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: AHYG.SI"}}}


$AHYG.SI: possibly delisted; no timezone found


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: IBOXIG"}}}


...

2026-05-25 00:04:05.933 | DEBUG    | hailmary.data.cache:get:38 - Cache hit key=da20efc4ae97


2026-05-25 00:04:06.011 | DEBUG    | hailmary.data.cache:get:38 - Cache hit key=b328b58cfa0a


2026-05-25 00:04:06.041 | DEBUG    | hailmary.data.cache:get:38 - Cache hit key=e443f1c4bd4f


98 ETFs, 96 with Yahoo data


,asset_class,name,ticker,wrapper,fund_manager,has_data,n_days,ytd_return,sharpe_1Y,ann_return_1Y,...,vol_3Y,sharpe_5Y,ann_return_5Y,max_dd_5Y,vol_5Y,corr_n,corr_book_YTD,corr_book_1Y,corr_book_3Y,corr_book_5Y
0,All Country World,iShares MSCI ACWI UCITS ETF,ISAC.L,UCITS (LSE),iShares,True,1577,0.101938,2.089687,0.286922,...,0.135394,0.791497,0.117310,-0.252319,0.155445,283,0.606386,0.396182,0.398714,0.398714
1,Artificial Intelligence,Xtrackers Artificial Intelligence & Big Data U...,XAID.L,?,Xtrackers,True,1577,0.209038,1.954498,0.445355,...,0.214673,0.690586,0.138193,-0.390967,0.223843,283,0.468201,0.328843,0.337274,0.337274
2,Asia ex-Japan,iShares MSCI All Country Asia ex Japan ETF,AAXJ,US,iShares,True,1561,0.242788,2.312403,0.545215,...,0.187810,0.622989,0.108974,-0.387568,0.197122,283,0.693988,0.678401,0.667684,0.667684
3,Asia High Yield USD Corporate Bonds *,iShares USD Asia High Yield Bond ETF,AHYG.SI,SG,iShares,False,0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
4,Australia,iShares MSCI Australia ETF,EWA,US,iShares,True,1561,0.104461,1.297645,0.226012,...,0.187422,0.628410,0.109493,-0.238233,0.195863,283,0.675043,0.718923,0.707973,0.707973
5,Aerospace & Defense,Invesco Aerospace & Defense ETF,PPA,US,Invesco,True,1561,0.104134,1.524784,0.304916,...,0.175459,1.191408,0.220938,-0.188185,0.181397,283,0.653833,0.694167,0.669270,0.669270
6,Battery Value-chain,L&G Battery Value-Chain UCITS ETF,BATT.L,?,LGIM,True,1577,0.336668,2.796159,1.204422,...,0.259137,0.654352,0.143440,-0.371618,0.254309,283,0.545360,0.380596,0.363301,0.363301
7,Biotechnology,iShares Biotechnology ETF,IBB,US,iShares,True,1561,-0.012310,1.585889,0.350155,...,0.198001,0.324875,0.048200,-0.364711,0.218168,283,0.656958,0.552391,0.530437,0.530437
8,Bitcoin (Accredited Investors only),Fidelity® Wise Origin® Bitcoin Fund,FBTC,?,Fidelity,True,575,-0.062571,-0.353863,-0.216277,...,0.504064,0.795431,0.315487,-0.465991,0.504064,283,0.818424,0.675312,0.672053,0.672053
9,Blockchain,Invesco CoinShares Global Blockchain UCITS ETF,BCHN.L,UCITS (LSE),Invesco,True,1575,0.201512,1.144133,0.475089,...,0.394984,0.325883,0.052735,-0.572249,0.384653,283,0.623292,0.368812,0.374313,0.374313


## Render HTML report

In [4]:
out = render_etf_explorer_report(
    explorer_df,
    REPORT_PATH,
    target_ann_return=TARGET_ANN_RETURN,
)
print(f'Wrote {out.resolve()}')

Wrote C:\Users\Dalva\src\project-hail-mary\reports\etf_explorer.html
